# Professional Exit Strategy Comparison

Testing more sophisticated exit approaches:

1. **Simple trailing stop** (current) - 30% from peak
2. **Take profit + stop loss** - Exit at +X%, stop at -Y%
3. **Profit-activated trail** - Trail only activates after +X% gain
4. **Scale out 2 tranches** - 50% at +50%, 50% trails
5. **Scale out 3 tranches** - 33% at +30%, +60%, +100%
6. **Tiered take profits** - 25% at each milestone
7. **Tighter trail after profit** - 30% trail, tightens to 15% after +50%

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from numba import njit
import warnings
warnings.filterwarnings('ignore')

print("Professional Exit Strategy Comparison 🎯")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()
df = df[df.index >= '2018-12-15'].dropna()

print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
# Entry signals (same for all)
entry_condition = (
    (df['sopr'] < 1) & 
    (df['sopr_sth'] < 1) & 
    (df['rl_zscore'] > 0.5)
)
entries = entry_condition & ~entry_condition.shift(1).fillna(False)
print(f"Entry signals: {entries.sum()}")

---
## Exit Strategy Functions

In [ ]:
@njit
def exit_simple_trail(price_arr, entry_idx, trail_pct=0.30):
    """Simple trailing stop from peak"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price > peak:
            peak = price
        if price <= peak * (1 - trail_pct):
            return j, price, 1.0  # exit_idx, exit_price, position_closed
    
    return len(price_arr) - 1, price_arr[-1], 1.0


@njit
def exit_tp_sl(price_arr, entry_idx, take_profit=0.50, stop_loss=0.20):
    """Fixed take profit + stop loss"""
    entry_price = price_arr[entry_idx]
    tp_price = entry_price * (1 + take_profit)
    sl_price = entry_price * (1 - stop_loss)
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price >= tp_price:
            return j, tp_price, 1.0
        if price <= sl_price:
            return j, sl_price, 1.0
    
    return len(price_arr) - 1, price_arr[-1], 1.0


@njit
def exit_profit_activated_trail(price_arr, entry_idx, profit_trigger=0.20, trail_pct=0.25, stop_loss=0.20):
    """Trail only activates after reaching profit threshold"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    trail_active = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        gain = (price - entry_price) / entry_price
        
        if price > peak:
            peak = price
        
        # Activate trail after profit threshold
        if not trail_active and gain >= profit_trigger:
            trail_active = True
        
        # Trail stop
        if trail_active and price <= peak * (1 - trail_pct):
            return j, price, 1.0
        
        # Stop loss before trail activates
        if not trail_active and gain <= -stop_loss:
            return j, price, 1.0
    
    return len(price_arr) - 1, price_arr[-1], 1.0


@njit
def exit_scale_out_2(price_arr, entry_idx, tp1=0.50, trail_pct=0.30, stop_loss=0.20):
    """
    Scale out 2 tranches:
    - 50% at +50% profit
    - 50% trails with 30% stop
    Returns weighted average exit
    """
    entry_price = price_arr[entry_idx]
    peak = entry_price
    tranche1_closed = False
    tranche1_exit = 0.0
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        gain = (price - entry_price) / entry_price
        
        if price > peak:
            peak = price
        
        # Tranche 1: Take profit at +50%
        if not tranche1_closed and gain >= tp1:
            tranche1_closed = True
            tranche1_exit = price
        
        # Tranche 2: Trail stop
        if price <= peak * (1 - trail_pct):
            if tranche1_closed:
                # Weighted average: 50% at tranche1, 50% at trail
                avg_exit = (tranche1_exit + price) / 2
                return j, avg_exit, 1.0
            else:
                return j, price, 1.0
        
        # Stop loss
        if gain <= -stop_loss:
            return j, price, 1.0
    
    # End of data
    if tranche1_closed:
        avg_exit = (tranche1_exit + price_arr[-1]) / 2
        return len(price_arr) - 1, avg_exit, 1.0
    return len(price_arr) - 1, price_arr[-1], 1.0


@njit
def exit_scale_out_3(price_arr, entry_idx, tp1=0.30, tp2=0.60, tp3=1.00, stop_loss=0.20):
    """
    Scale out 3 tranches:
    - 33% at +30%
    - 33% at +60%
    - 33% at +100% or trail
    """
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    t1_closed, t2_closed, t3_closed = False, False, False
    t1_exit, t2_exit, t3_exit = 0.0, 0.0, 0.0
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        gain = (price - entry_price) / entry_price
        
        if price > peak:
            peak = price
        
        # Tranches
        if not t1_closed and gain >= tp1:
            t1_closed = True
            t1_exit = price
        
        if not t2_closed and gain >= tp2:
            t2_closed = True
            t2_exit = price
        
        if not t3_closed and gain >= tp3:
            t3_closed = True
            t3_exit = price
            # All tranches closed
            avg_exit = (t1_exit + t2_exit + t3_exit) / 3
            return j, avg_exit, 1.0
        
        # Trail stop for remaining position (30% from peak)
        if price <= peak * 0.70:
            exits = []
            if t1_closed: exits.append(t1_exit)
            if t2_closed: exits.append(t2_exit)
            exits.append(price)  # Remaining at trail
            
            # Weight equally
            avg_exit = sum(exits) / len(exits) if exits else price
            return j, avg_exit, 1.0
        
        # Stop loss
        if gain <= -stop_loss:
            return j, price, 1.0
    
    # End of data
    exits = []
    if t1_closed: exits.append(t1_exit)
    if t2_closed: exits.append(t2_exit)
    if t3_closed: exits.append(t3_exit)
    exits.append(price_arr[-1])  # Remaining
    avg_exit = sum(exits) / len(exits)
    return len(price_arr) - 1, avg_exit, 1.0


@njit
def exit_tiered_tp(price_arr, entry_idx, stop_loss=0.20):
    """
    Tiered take profits:
    - 25% at +25%
    - 25% at +50%
    - 25% at +75%
    - 25% at +100%
    With 20% stop loss
    """
    entry_price = price_arr[entry_idx]
    tiers = [0.25, 0.50, 0.75, 1.00]
    tier_exits = [0.0, 0.0, 0.0, 0.0]
    tiers_hit = 0
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        gain = (price - entry_price) / entry_price
        
        # Check tiers
        while tiers_hit < 4 and gain >= tiers[tiers_hit]:
            tier_exits[tiers_hit] = price
            tiers_hit += 1
        
        # All tiers hit
        if tiers_hit == 4:
            avg_exit = sum(tier_exits) / 4
            return j, avg_exit, 1.0
        
        # Stop loss
        if gain <= -stop_loss:
            if tiers_hit > 0:
                # Partial profit locked
                exits = tier_exits[:tiers_hit] + [price] * (4 - tiers_hit)
                avg_exit = sum(exits) / 4
                return j, avg_exit, 1.0
            return j, price, 1.0
    
    # End of data
    if tiers_hit > 0:
        exits = tier_exits[:tiers_hit] + [price_arr[-1]] * (4 - tiers_hit)
        avg_exit = sum(exits) / 4
        return len(price_arr) - 1, avg_exit, 1.0
    return len(price_arr) - 1, price_arr[-1], 1.0


@njit
def exit_tightening_trail(price_arr, entry_idx, initial_trail=0.30, tight_trail=0.15, profit_threshold=0.50, stop_loss=0.20):
    """
    Trail tightens after profit threshold:
    - 30% trail initially
    - Tightens to 15% after +50% gain
    """
    entry_price = price_arr[entry_idx]
    peak = entry_price
    trail_pct = initial_trail
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        gain = (price - entry_price) / entry_price
        
        if price > peak:
            peak = price
        
        # Tighten trail after profit threshold
        if gain >= profit_threshold:
            trail_pct = tight_trail
        
        # Trail stop
        if price <= peak * (1 - trail_pct):
            return j, price, 1.0
        
        # Stop loss
        if gain <= -stop_loss:
            return j, price, 1.0
    
    return len(price_arr) - 1, price_arr[-1], 1.0

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, fees=0.001, **kwargs):
    """Run backtest with given exit function"""
    price_arr = df['price'].values
    dates = df.index
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        exit_idx, exit_price, _ = exit_func(price_arr, entry_idx, **kwargs)
        
        entry_price = price_arr[entry_idx]
        gross_return = (exit_price / entry_price) - 1
        net_return = gross_return - (2 * fees)
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    
    return trades_df


def calc_metrics(trades, initial_capital, df):
    """Calculate performance metrics"""
    if len(trades) == 0:
        return {'total_return': 0, 'sharpe': 0, 'max_dd': 0, 'win_rate': 0}
    
    final_equity = trades['equity'].iloc[-1]
    total_return = (final_equity / initial_capital) - 1
    
    years = (trades['exit_date'].iloc[-1] - trades['entry_date'].iloc[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0
    
    win_rate = (trades['net_return'] > 0).mean()
    
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(trades)/years) if returns.std() > 0 else 0
    
    # Max drawdown
    equity = [initial_capital] + list(trades['equity'])
    peak = equity[0]
    max_dd = 0
    for eq in equity:
        if eq > peak: peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd: max_dd = dd
    
    # Buy & hold
    bh_return = (df['price'].iloc[-1] / df.loc[trades['entry_date'].iloc[0], 'price']) - 1
    
    # Profit factor
    winners = trades[trades['net_return'] > 0]['net_return'].sum()
    losers = abs(trades[trades['net_return'] <= 0]['net_return'].sum())
    pf = winners / losers if losers > 0 else float('inf')
    
    return {
        'total_return': total_return,
        'cagr': cagr,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'win_rate': win_rate,
        'profit_factor': pf,
        'n_trades': len(trades),
        'bh_return': bh_return,
        'final_equity': final_equity
    }

---
## Run All Exit Strategies

In [ ]:
INITIAL = 100000

strategies = [
    ('1. Simple 30% Trail', exit_simple_trail, {'trail_pct': 0.30}),
    ('2. TP 50% / SL 20%', exit_tp_sl, {'take_profit': 0.50, 'stop_loss': 0.20}),
    ('3. TP 100% / SL 20%', exit_tp_sl, {'take_profit': 1.00, 'stop_loss': 0.20}),
    ('4. TP 150% / SL 20%', exit_tp_sl, {'take_profit': 1.50, 'stop_loss': 0.20}),
    ('5. Profit-Activated Trail (20%→25%)', exit_profit_activated_trail, {'profit_trigger': 0.20, 'trail_pct': 0.25, 'stop_loss': 0.20}),
    ('6. Profit-Activated Trail (30%→20%)', exit_profit_activated_trail, {'profit_trigger': 0.30, 'trail_pct': 0.20, 'stop_loss': 0.20}),
    ('7. Scale Out 2x (50% at +50%)', exit_scale_out_2, {'tp1': 0.50, 'trail_pct': 0.30, 'stop_loss': 0.20}),
    ('8. Scale Out 2x (50% at +100%)', exit_scale_out_2, {'tp1': 1.00, 'trail_pct': 0.30, 'stop_loss': 0.20}),
    ('9. Scale Out 3x (+30/60/100%)', exit_scale_out_3, {'tp1': 0.30, 'tp2': 0.60, 'tp3': 1.00, 'stop_loss': 0.20}),
    ('10. Scale Out 3x (+50/100/150%)', exit_scale_out_3, {'tp1': 0.50, 'tp2': 1.00, 'tp3': 1.50, 'stop_loss': 0.20}),
    ('11. Tiered TP (25/50/75/100%)', exit_tiered_tp, {'stop_loss': 0.20}),
    ('12. Tightening Trail (30%→15%)', exit_tightening_trail, {'initial_trail': 0.30, 'tight_trail': 0.15, 'profit_threshold': 0.50, 'stop_loss': 0.20}),
]

results = []

print("EXIT STRATEGY COMPARISON")
print("="*140)
print(f"{'Strategy':<35} {'Return':>12} {'CAGR':>10} {'Sharpe':>8} {'Win%':>8} {'PF':>8} {'MaxDD':>10} {'Trades':>8} {'Final $':>14}")
print("-"*140)

for name, func, kwargs in strategies:
    trades = run_backtest(df, entries, func, INITIAL, **kwargs)
    m = calc_metrics(trades, INITIAL, df)
    
    print(f"{name:<35} {m['total_return']*100:>+11.0f}% {m['cagr']*100:>+9.1f}% "
          f"{m['sharpe']:>8.2f} {m['win_rate']*100:>7.0f}% {m['profit_factor']:>8.2f} "
          f"{m['max_dd']*100:>9.0f}% {m['n_trades']:>8} ${m['final_equity']:>13,.0f}")
    
    results.append({'name': name, 'trades': trades, 'metrics': m})

print("-"*140)
print(f"{'Buy & Hold':<35} {m['bh_return']*100:>+11.0f}%")

In [ ]:
# Find best by different criteria
print("\n" + "="*60)
print("RANKINGS")
print("="*60)

# Best by total return
by_return = sorted(results, key=lambda x: x['metrics']['total_return'], reverse=True)
print(f"\n📈 Best by Total Return:")
for i, r in enumerate(by_return[:3]):
    print(f"   {i+1}. {r['name']}: {r['metrics']['total_return']*100:+,.0f}%")

# Best by Sharpe
by_sharpe = sorted(results, key=lambda x: x['metrics']['sharpe'], reverse=True)
print(f"\n📊 Best by Sharpe Ratio:")
for i, r in enumerate(by_sharpe[:3]):
    print(f"   {i+1}. {r['name']}: {r['metrics']['sharpe']:.2f}")

# Best by Win Rate
by_wr = sorted(results, key=lambda x: x['metrics']['win_rate'], reverse=True)
print(f"\n🎯 Best by Win Rate:")
for i, r in enumerate(by_wr[:3]):
    print(f"   {i+1}. {r['name']}: {r['metrics']['win_rate']*100:.0f}%")

# Lowest drawdown
by_dd = sorted(results, key=lambda x: x['metrics']['max_dd'], reverse=True)
print(f"\n🛡️ Lowest Max Drawdown:")
for i, r in enumerate(by_dd[:3]):
    print(f"   {i+1}. {r['name']}: {r['metrics']['max_dd']*100:.0f}%")

In [ ]:
# Trade details for top strategies
print("\n" + "="*100)
print("TRADE DETAILS - TOP 3 BY RETURN")
print("="*100)

for r in by_return[:3]:
    print(f"\n{r['name']}")
    print("-"*80)
    trades = r['trades']
    for _, t in trades.iterrows():
        print(f"  {t['entry_date'].date()} → {t['exit_date'].date()}: "
              f"${t['entry_price']:,.0f} → ${t['exit_price']:,.0f} = {t['net_return']*100:+.0f}% "
              f"({t['days_held']} days)")

In [ ]:
# Visualization
import plotly.graph_objects as go

fig = go.Figure()

# Bar chart comparing strategies
names = [r['name'].replace('1. ', '').replace('2. ', '').replace('3. ', '') for r in results]
returns = [r['metrics']['total_return'] * 100 for r in results]
sharpes = [r['metrics']['sharpe'] for r in results]

fig.add_trace(go.Bar(
    x=names,
    y=returns,
    name='Return %',
    marker_color='blue',
    text=[f"{r:+,.0f}%" for r in returns],
    textposition='outside'
))

fig.update_layout(
    title='Exit Strategy Returns Comparison',
    yaxis_title='Total Return %',
    height=500,
    xaxis_tickangle=-45
)
fig.show()

In [ ]:
# Risk-adjusted comparison
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[r['metrics']['max_dd'] * -100 for r in results],
    y=[r['metrics']['total_return'] * 100 for r in results],
    mode='markers+text',
    text=[r['name'].split('.')[0] for r in results],
    textposition='top center',
    marker=dict(size=15, color=[r['metrics']['sharpe'] for r in results], colorscale='Viridis', showscale=True)
))

fig.update_layout(
    title='Return vs Risk (color = Sharpe)',
    xaxis_title='Max Drawdown %',
    yaxis_title='Total Return %',
    height=500
)
fig.show()

---
## Summary & Recommendation

In [ ]:
# Find best overall (balance of return and risk)
# Score = Return * Sharpe / |MaxDD|
for r in results:
    m = r['metrics']
    r['score'] = (m['total_return'] * m['sharpe']) / abs(m['max_dd']) if m['max_dd'] != 0 else 0

by_score = sorted(results, key=lambda x: x['score'], reverse=True)

print("\n" + "="*60)
print("OVERALL RANKING (Return × Sharpe / MaxDD)")
print("="*60)
for i, r in enumerate(by_score[:5]):
    m = r['metrics']
    print(f"\n{i+1}. {r['name']}")
    print(f"   Return: {m['total_return']*100:+,.0f}%")
    print(f"   Sharpe: {m['sharpe']:.2f}")
    print(f"   Max DD: {m['max_dd']*100:.0f}%")
    print(f"   Win Rate: {m['win_rate']*100:.0f}%")
    print(f"   Score: {r['score']:.2f}")